# {title}

**状态：{status}。** 本 Notebook 使用官方 `AutoModel` 和 `inference_instruct2`，在 Colab GPU 上合成单句并直接试听。单次成功不代表音质或稳定性能评测。

[官方代码]({source_url}) · [模型页]({model_url}) · 许可：{license}。模型约 9.8 GB，首次下载前建议至少 18 GiB 可用磁盘。源码 commit `074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc`；模型 revision `3a9eb0e104c9ce5bb7ef0af4ac86d9b79abc4029`。


## 1. 检查 Colab GPU 与空间
请在“运行时 → 更改运行时类型”选择 GPU。

In [ ]:
import os, platform, shutil, subprocess, sys
from pathlib import Path

assert Path('/content').exists(), '请在 Google Colab 中运行'
assert shutil.which('nvidia-smi'), '请先启用 Colab GPU'
MODEL_ROOT = Path('/content/models')
MODEL_DIR = MODEL_ROOT / 'FunAudioLLM' / 'Fun-CosyVoice3-0.5B-2512'
REPO = Path('/content/CosyVoice')
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['MODELSCOPE_CACHE'] = str(MODEL_ROOT / 'modelscope_cache')
os.environ['HF_HOME'] = str(MODEL_ROOT / 'huggingface_cache')
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))
print('Python:', sys.version)
print('Platform:', platform.platform())
free_gib = shutil.disk_usage(MODEL_ROOT).free / 2**30
print('Model directory:', MODEL_DIR, 'free GiB:', round(free_gib, 1))
assert free_gib >= 18 or (MODEL_DIR / 'cosyvoice3.yaml').exists(), '首次安装与下载建议至少 18 GiB 可用空间'


## 2. 安装缺少的依赖
复用 Colab 自带的 PyTorch、torchaudio 和 CUDA，不安装 cu126 或 vLLM。固定现有 torch 版本；若依赖冲突，停止并查看本单元输出。

In [ ]:
import torch, torchaudio
assert torch.cuda.is_available(), 'PyTorch 未检测到 GPU'
TORCH_VERSION = torch.__version__
AUDIO_VERSION = torchaudio.__version__
print('Using torch:', TORCH_VERSION, 'CUDA:', torch.version.cuda)
print('Using torchaudio:', AUDIO_VERSION)
packages = [
    f'torch=={TORCH_VERSION}', f'torchaudio=={AUDIO_VERSION}',
    'transformers==4.51.3', 'conformer==0.3.2', 'diffusers==0.29.0',
    'hydra-core==1.3.2', 'HyperPyYAML==1.2.3', 'inflect==7.3.1',
    'librosa>=0.10.2,<1', 'modelscope>=1.20,<2', 'onnxruntime-gpu>=1.22,<2',
    'omegaconf>=2.3,<3', 'openai-whisper==20250625', 'soundfile>=0.12,<1',
    'wetext==0.0.4', 'x-transformers==2.11.24',
    'gdown>=5,<6', 'matplotlib>=3.7,<4', 'wget==3.2',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)


## 3. 获取固定版本的官方源码与模型
源码只用于调用官方推理接口。模型明确下载到 `/content/models`；重复运行会复用已有文件。

In [ ]:
CODE_REVISION = '074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc'
MODEL_REVISION = '3a9eb0e104c9ce5bb7ef0af4ac86d9b79abc4029'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/FunAudioLLM/CosyVoice.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', CODE_REVISION], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive', 'third_party/Matcha-TTS'], check=True)
print('Source revision:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
from modelscope import snapshot_download
snapshot_download('FunAudioLLM/Fun-CosyVoice3-0.5B-2512', revision=MODEL_REVISION, local_dir=str(MODEL_DIR))
assert (MODEL_DIR / 'cosyvoice3.yaml').is_file()
print('Model revision:', MODEL_REVISION, 'directory:', MODEL_DIR)


## 4. 核对模型文件
下列 SHA-256 与固定 ModelScope revision 的文件清单一致。若有不匹配，停止推理并保存实际哈希。

In [ ]:
import hashlib
expected_sha256 = {
    'llm.pt': 'd9565e41454e860768447f4ec5c5244a06b1ccdffdc7890fa593d907c93eebcc',
    'flow.pt': 'a6fab32a7825e5b0bc855ddd948f8db9370b0a786fbc249caa4595e95b608e4b',
    'hift.pt': 'b279d7641eb97ae55b3b540cfba4f953c26492a2df758328a89a4d007ab87a65',
    'speech_tokenizer_v3.onnx': '23236a74175dbdda47afc66dbadd5bcb41303c467a57c261cb8539ad9db9208d',
    'flow.decoder.estimator.fp32.onnx': '9b51b9533a55937762b262bf2cf9c6220ce40760f76d6532cb16a6a6d84059a8',
    'CosyVoice-BlankEN/model.safetensors': '130282af0dfa9fe5840737cc49a0d339d06075f83c5a315c3372c9a0740d0b96',
}
for relative, expected in expected_sha256.items():
    path = MODEL_DIR / relative
    assert path.is_file(), f'缺少模型文件: {path}'
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    actual = digest.hexdigest()
    assert actual == expected, f'SHA-256 不一致: {relative}; actual={actual}; expected={expected}; bytes={path.stat().st_size}'
    print(relative, 'SHA-256 OK')


## 5. 输入文本并加载模型
修改 `TEXT` 即可合成自己的句子。参考音频使用官方源码中的示例。

In [ ]:
TEXT = '你好，欢迎体验 CosyVoice3。'
assert TEXT.strip(), '请输入非空文本'
import importlib.metadata as metadata
import soundfile as sf
import numpy as np
sys.path.insert(0, str(REPO / 'third_party/Matcha-TTS'))
sys.path.insert(0, str(REPO))
from cosyvoice.cli.cosyvoice import AutoModel
print('Text:', TEXT)
for package in ('torch', 'torchaudio', 'transformers', 'modelscope', 'onnxruntime-gpu', 'openai-whisper', 'soundfile'):
    print(package, metadata.version(package))
assert metadata.version('torch') == TORCH_VERSION
assert metadata.version('torchaudio') == AUDIO_VERSION
model = AutoModel(model_dir=str(MODEL_DIR), load_vllm=False, load_trt=False, fp16=False)
print('Model loaded; sample rate:', model.sample_rate)


## 6. 合成、试听与记录结果
单次计时包含 `inference_instruct2` 调用与输出拼接，不含安装、下载、加载和 WAV 写入。同步 CUDA 后计时；RTF 只对应这一次输入。

In [ ]:
import time
from IPython.display import Audio, display
from google.colab import files

PROMPT = REPO / 'asset/zero_shot_prompt.wav'
assert PROMPT.is_file(), PROMPT
INSTRUCTION = 'You are a helpful assistant. 请用自然、清晰的语气朗读。<|endofprompt|>'
torch.manual_seed(1986)
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
chunks = [item['tts_speech'].detach().cpu() for item in model.inference_instruct2(
    TEXT, INSTRUCTION, str(PROMPT), stream=False, speed=1.0, text_frontend=True
)]
torch.cuda.synchronize()
elapsed_s = time.perf_counter() - start
assert chunks, '模型没有返回音频'
audio = torch.cat(chunks, dim=-1).squeeze().numpy().astype(np.float32, copy=False)
output_path = Path('/content/cosyvoice3_output.wav')
sf.write(output_path, audio, int(model.sample_rate), format='WAV', subtype='PCM_16')
audio_s = len(audio) / model.sample_rate
print('Input characters:', len(TEXT), 'Audio seconds:', round(audio_s, 3))
print('Single-call elapsed s:', round(elapsed_s, 3), 'RTF:', round(elapsed_s / audio_s, 3))
print('GPU peak allocated MiB:', round(torch.cuda.max_memory_allocated() / 2**20, 1))
print('GPU peak reserved MiB:', round(torch.cuda.max_memory_reserved() / 2**20, 1))
print('Output SHA-256:', hashlib.sha256(output_path.read_bytes()).hexdigest())
display(Audio(filename=str(output_path)))
files.download(str(output_path))


## 验证边界
仓库状态：**{status}**。只有实际 Colab GPU 合成成功并保存运行输出后，才可更新状态。单句结果不证明音质、稳定性能或其他运行环境的适配。